In [1]:
import pandas as pd
import re
import unicodedata
from collections import Counter

# Load CSV (tab-separated)
df = pd.read_csv("AbsArSumCorpus_csv_v1.zip", sep="\t", names=["article", "summary"])

df = df.dropna().drop_duplicates().reset_index(drop=True)

print("Shape:", df.shape)
print(df.head(3))

Shape: (49582, 2)
                                             article  \
0  حقق حزب "البديل من أجل ألمانيا" اليميني الشعبو...   
1  بدأت اليوم الجمعة( 23 أيلول/ سبتمبر 2016 ) في ...   
2  قال مسؤولون إن شخصا أصيب إصابة بالغة بعد تعرضه...   

                                             summary  
0  تشير آخر استطلاعات الرأي الألمانية إلى تقدم حز...  
1  بدأت محكمة ميونخ النظر في اتهامات متعلقة برجل ...  
2  أعلن حاكم كارولاينا الشمالية حالة الطورائ في م...  


In [2]:
ARABIC_RE = re.compile(r'[\u0600-\u06FF]')
ENGLISH_RE = re.compile(r'[A-Za-z]')
HTML_RE = re.compile(r'<[^>]+>')
URL_RE = re.compile(r'https?://\S+|www\.\S+')
DIACRITICS_RE = re.compile(r'[\u064B-\u0652]')
TATWEEL_RE = re.compile(r'ـ')
DIGIT_RE = re.compile(r'\d')
REPEATED_PUNCT_RE = re.compile(r'([!?؟،؛.,:])\1+')

def inspect(text):
    text = str(text)

    return {
        "len": len(text),
        "words": len(text.split()),
        "has_html": bool(HTML_RE.search(text)),
        "has_url": bool(URL_RE.search(text)),
        "has_diacritics": bool(DIACRITICS_RE.search(text)),
        "has_tatweel": bool(TATWEEL_RE.search(text)),
        "has_english": bool(ENGLISH_RE.search(text)),
        "has_digits": bool(DIGIT_RE.search(text)),
        "repeated_punct": bool(REPEATED_PUNCT_RE.search(text)),
        "arabic_ratio": len(ARABIC_RE.findall(text)) / (len(text)+1e-6),
    }

In [3]:
article_diag = df["article"].apply(inspect).apply(pd.Series).add_prefix("article_")
summary_diag = df["summary"].apply(inspect).apply(pd.Series).add_prefix("summary_")

scan = pd.concat([df, article_diag, summary_diag], axis=1)

In [4]:
summary_stats = {
    "rows": len(scan),

    "article_html": scan["article_has_html"].sum(),
    "summary_html": scan["summary_has_html"].sum(),

    "article_url": scan["article_has_url"].sum(),
    "summary_url": scan["summary_has_url"].sum(),

    "article_diacritics": scan["article_has_diacritics"].sum(),
    "summary_diacritics": scan["summary_has_diacritics"].sum(),

    "article_tatweel": scan["article_has_tatweel"].sum(),
    "summary_tatweel": scan["summary_has_tatweel"].sum(),

    "article_english": scan["article_has_english"].sum(),
    "summary_english": scan["summary_has_english"].sum(),

    "article_digits": scan["article_has_digits"].sum(),
    "summary_digits": scan["summary_has_digits"].sum(),
}

for k, v in summary_stats.items():
    print(k, ":", v)

rows : 49582
article_html : 0
summary_html : 0
article_url : 123
summary_url : 0
article_diacritics : 28614
summary_diacritics : 9623
article_tatweel : 14530
summary_tatweel : 2343
article_english : 9758
summary_english : 1414
article_digits : 45954
summary_digits : 9085


In [5]:
print("\nArticle length stats:")
print(scan["article_words"].describe(percentiles=[0.5, 0.9, 0.95, 0.99]))

print("\nSummary length stats:")
print(scan["summary_words"].describe(percentiles=[0.5, 0.9, 0.95, 0.99]))


Article length stats:
count    49582.000000
mean       376.485418
std        223.050954
min         33.000000
50%        314.000000
90%        697.000000
95%        823.000000
99%       1081.000000
max       2898.000000
Name: article_words, dtype: float64

Summary length stats:
count    49582.000000
mean        33.537191
std          3.732815
min          2.000000
50%         34.000000
90%         38.000000
95%         39.000000
99%         40.000000
max         45.000000
Name: summary_words, dtype: float64


In [6]:
def show(df, condition, n=3):
    sample = df[condition].head(n)
    for i, row in sample.iterrows():
        print("\n---")
        print("ARTICLE:\n", row["article"][:800])
        print("\nSUMMARY:\n", row["summary"][:400])

# Check English
show(scan, scan["article_has_english"], 3)

# Check digits
show(scan, scan["article_has_digits"], 3)

# Check URLs
show(scan, scan["article_has_url"], 3)


---
ARTICLE:
 حقق حزب "البديل من أجل ألمانيا" اليميني الشعبوي أفضل نتائجه في أحدث استطلاعات الرأي بحصوله على تأييد 16% من الألمان في جميع أنحاء ألمانيا. وأظهر الاستطلاع الذي أجراه معهد "إنفراتست ديماب" بتكليف من شبكة (ARD)إيه آر دي الإعلامية الألمانية أن شعبية الحزب ارتفعت بذلك بنسبة 2% مقارنة باستطلاع مماثل أجري مطلع الشهر الجاري لشبكة(ZDF) تسي دي إف الإخبارية. وحصل التحالف المسيحي، المنتمية إليه المستشارة أنغيلا ميركل، في الاستطلاع الذي نشرت نتائجه اليوم الجمعة (23أيلول/ سبتمبر 2016 )على نسبة 32% بتراجع قدره 1% عن الاستطلاع السابق. كما تراجعت شعبية الحزب الاشتراكي الديمقراطي، الشريك في الائتلاف الحاكم، بنفس النسبة ليحصل على نسبة 22%. وفي المقابل، ارتفعت شعبية حزب الخضر بمقدار 1% لتصل إلى 12%، بينما تراجعت شعبية حزب "اليسار" بنسبة 1% ليحصل على نسبة 8%، في حين ارتفعت شعبية الحزب الديمقراطي الحر بنسبة 1%

SUMMARY:
 تشير آخر استطلاعات الرأي الألمانية إلى تقدم حزب البديل اليميني في حين يستمر تراجع الحزب المسيحي الديمقراطي الذي تقوده المستشارة ميركل. وصرحت مصادر رسمية أن النتائج باتت غير 

In [7]:
def clean_arasum(text):
    if not isinstance(text, str):
        return ""

    text = text.strip()

    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)

    text = re.sub(r'[\u0617-\u061A\u064B-\u0652\u0670\u0640]', '', text)

    text = re.sub(r'[إأآ]', 'ا', text)
    text = re.sub(r'ى', 'ي', text)

    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [8]:
df = df.dropna().drop_duplicates().reset_index(drop=True)

# Apply light cleaning
df["article"] = df["article"].apply(clean_arasum)
df["summary"] = df["summary"].apply(clean_arasum)

# Remove rows that became empty after cleaning
df = df[
    (df["article"].str.strip() != "") &
    (df["summary"].str.strip() != "")
].reset_index(drop=True)

# Add Seq2Seq markers
df["summary"] = df["summary"].apply(lambda x: f"<start> {x} <end>")

df.to_csv("cleaned_arasum_final.csv", index=False, encoding="utf-8-sig")